In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [2]:
import numpy as np
#SEED = 45
#np.random.seed(SEED)

from models import GibbsSamplerLLFM
from data import generate_parametric
from post_process import frequency_align, hungarian_align
from sklearn.metrics import mean_squared_error

In [3]:
T = 500
S = 4
K = 8
alpha = 5.0
rho = 0.2
sigma_w = 1.0
mu_b = [0, 0, 0, 0]
sigma_b = 1.0
n_iter = 1500
burn = 500
n_subsample = 200
Y, Z_true, W_true, b_true, p_true = generate_parametric(T=T, S=S, K=K, alpha=alpha, rho=rho, sigma_w=sigma_w, mu_b=mu_b, sigma_b=sigma_b)
print("Latent probabilities:", p_true)
print("True weight matrix W_true:", W_true)
print("True bias vector b_true:", b_true)

Latent probabilities: [0.08401565 0.82596942 0.63892844 0.04666309 0.47825644 0.01248382
 0.87761462 0.96387164]
True weight matrix W_true: [[ 0.         -1.66827963 -0.          0.        ]
 [ 0.         -0.         -0.          0.        ]
 [-0.         -0.          0.         -0.        ]
 [ 0.          0.          0.11877061  0.        ]
 [-0.         -0.          0.          0.        ]
 [ 3.21706603  0.          0.          0.        ]
 [ 0.          0.         -0.83431273 -0.        ]
 [-0.         -0.77404092  0.15735996  0.        ]]
True bias vector b_true: [ 0.78089457 -1.25705881  0.99879224 -1.28823581]


In [4]:
np.linalg.norm(W_true, ord='fro')

np.float64(3.8035189559046345)

In [5]:
sampler = GibbsSamplerLLFM(
    Data=Y,
    K=K,
    alpha=alpha,
    rho=rho,
    sigma_w=sigma_w,
    mu_b=mu_b,
    sigma_b=sigma_b,
    n_iter=n_iter,
    burn=burn,
    n_subsample=n_subsample
)

sampler.run(verbose=True)
sampler.get_posterior_samples()

Iteration 25: ||W||_F = 2.9331
Iteration 50: ||W||_F = 2.5993
Iteration 75: ||W||_F = 3.4401
Iteration 100: ||W||_F = 2.5687
Iteration 125: ||W||_F = 2.9527
Iteration 150: ||W||_F = 2.8059
Iteration 175: ||W||_F = 1.7986
Iteration 200: ||W||_F = 1.9845
Iteration 225: ||W||_F = 2.8070
Iteration 250: ||W||_F = 3.0275
Iteration 275: ||W||_F = 1.1707
Iteration 300: ||W||_F = 1.6657
Iteration 325: ||W||_F = 1.3274
Iteration 350: ||W||_F = 3.5120
Iteration 375: ||W||_F = 2.1677
Iteration 400: ||W||_F = 3.4866
Iteration 425: ||W||_F = 3.4467
Iteration 450: ||W||_F = 1.6997
Iteration 475: ||W||_F = 2.9409
Iteration 500: ||W||_F = 1.6931
Iteration 525: ||W||_F = 3.6996
Iteration 550: ||W||_F = 2.9591
Iteration 575: ||W||_F = 2.8022
Iteration 600: ||W||_F = 2.0704
Iteration 625: ||W||_F = 2.0785
Iteration 650: ||W||_F = 1.7161
Iteration 675: ||W||_F = 2.2306
Iteration 700: ||W||_F = 3.6889
Iteration 725: ||W||_F = 2.9979
Iteration 750: ||W||_F = 3.3425
Iteration 775: ||W||_F = 2.6633
Iteration 8

In [6]:
Z_samples = sampler.good_samples_Z      # (n_samples, T, K)
W_samples = sampler.good_samples_W      # (n_samples, K, S)
b_samples = sampler.good_samples_b      # (n_samples, S)


Z_aligned_samples, W_aligned_samples, b_aligned_samples = hungarian_align(Z_samples, W_samples, b_samples, W_true)

W_mean = W_aligned_samples.mean(axis=0)
Z_mean = Z_aligned_samples.mean(axis=0)
p_mean = Z_mean.mean(axis=0)
b_mean = b_aligned_samples.mean(axis=0)

In [7]:
print("True bias vector b_true:", b_true)
print("Estimated bias vector b_mean:", b_mean)
print("True weight matrix W_true:", W_true)
print("Estimated weight matrix W_aligned:", W_mean)
print("Latent probabilities p_true:", p_true)
print("Estimated latent probabilities p_aligned:", p_mean)

True bias vector b_true: [ 0.78089457 -1.25705881  0.99879224 -1.28823581]
Estimated bias vector b_mean: [ 0.04863233 -1.45420137 -0.01963152 -0.7175812 ]
True weight matrix W_true: [[ 0.         -1.66827963 -0.          0.        ]
 [ 0.         -0.         -0.          0.        ]
 [-0.         -0.          0.         -0.        ]
 [ 0.          0.          0.11877061  0.        ]
 [-0.         -0.          0.          0.        ]
 [ 3.21706603  0.          0.          0.        ]
 [ 0.          0.         -0.83431273 -0.        ]
 [-0.         -0.77404092  0.15735996  0.        ]]
Estimated weight matrix W_aligned: [[ 8.56832352e-02 -1.07207131e+00  1.76503236e-01 -8.17834444e-02]
 [ 3.19330367e-03  0.00000000e+00 -8.49637419e-06 -5.30248434e-03]
 [ 6.11705378e-03 -1.64181536e-03 -3.53148471e-03  1.43889102e-02]
 [-1.60429521e-02  2.50233173e-02  1.58654660e-01 -4.62251928e-02]
 [-1.22327484e-02  2.05212896e-03 -1.03935783e-02 -9.57150878e-03]
 [ 1.06303878e+00 -6.36525530e-02  2.24

In [8]:
print("W elementwise MSE:")
print((W_true - W_mean)**2)

print("Bias elementwise MSE:")
print((b_true - b_mean)**2)

print("p elementwise MSE:")
print((p_true - p_mean)**2)

W elementwise MSE:
[[7.34161679e-03 3.55464367e-01 3.11533922e-02 6.68853178e-03]
 [1.01971883e-05 0.00000000e+00 7.21883743e-11 2.81163402e-05]
 [3.74183470e-05 2.69555766e-06 1.24713843e-05 2.07040737e-04]
 [2.57376313e-04 6.26166411e-04 1.59073731e-03 2.13676845e-03]
 [1.49640133e-04 4.21123328e-06 1.08026470e-04 9.16137804e-05]
 [4.63983340e+00 4.05164750e-03 5.04785233e-02 5.10447623e-02]
 [5.64220744e-03 2.94440456e-04 2.69121325e-01 1.72965134e-02]
 [3.20818339e-05 3.38693962e-01 8.05999528e-02 5.78906586e-03]]
Bias elementwise MSE:
[0.53620799 0.03886519 1.03718696 0.32564669]
p elementwise MSE:
[0.36681719 0.2914266  0.15478593 0.14930257 0.01637797 0.43060654
 0.17696714 0.21134399]


In [9]:
print("W MSE:", mean_squared_error(W_true, W_mean))
print("p MSE:", mean_squared_error(p_true, p_mean))
print("Bias MSE:", mean_squared_error(b_true, b_mean))


W MSE: 0.18339963348634625
p MSE: 0.22470349263719164
Bias MSE: 0.48447670820992594


In [10]:

def normalized_mse(true, est):
    return np.sum((true - est)**2) / np.sum(true**2)

print("W NMSE:", normalized_mse(W_true, W_mean))
print("Bias NMSE:", normalized_mse(b_true, b_mean))
print("p NMSE:", normalized_mse(p_true, p_mean))

W NMSE: 0.405674090988412
Bias NMSE: 0.39980495072994465
p NMSE: 0.593701318513993


In [11]:
def corrcoef_flat(a, b):
    return np.corrcoef(a.flatten(), b.flatten())[0,1]

print("W correlation:", corrcoef_flat(W_true, W_mean))
print("Bias correlation:", np.corrcoef(b_true, b_mean)[0,1])
print("p correlation:", np.corrcoef(p_true, p_mean)[0,1])

W correlation: 0.894113653206877
Bias correlation: 0.8936528105422373
p correlation: -0.55190482947045
